# Extracción de datos de la Temperatura Superficial del Mar Peruano: NOAA (National Oceanic and Atmospheric Administration)

La agencia científica de Estados Unidos provee un archivo semanal maestro listo para consumir y extraer los datos oficiales de las temperaturas del mar, esta misma fuente es la que utiliza el SENAHMI y el ENFEN para construir sus gráficos de monitoreo, por lo que no se recurrirá la búsqueda en páginas gubernamentales que podrían estar desactualizadas o tomadas en distintos periodos.


Para este proyecto se extraerán los datos de la región comprendida en la región Niño 1+2 (el cuadrante oceánico 0–10°S, 90°W-80°W que abarca directamente la costa de Piura y Tumbes).

Los registros rescatados serán de utilidad para manejar las anomalías de casos de dengue generadas por el fenómeno del Niño en 2017 y 2023

## Importación de Librerías
A continuación se describe el uso de cada librería:

requests: Permite extraer los datos de la NOAA de forma local utilizando el URL oficial.

re: El archivo .for tiene un formato legado de ancho fijo (heredado de sistemas Fortran) donde la temperatura absoluta y la anomalía térmica a veces vienen pegadas sin espacio (ej. 26.5-0.4 o 27.8 1.5). Se usará una expresión regular (regex) para separar limpiamente ambas lecturas y mapearlas a la semana ISO.

pandas: Estructuración y conversión en semanas epidemiológicas.

In [1]:
import requests
import re
import pandas as pd

## Conexión y extracción de datos

In [2]:
# 1. CONEXIÓN AL SERVIDOR DE LA NOAA (Fuente oficial de los informes técnicos ENFEN)
url_noaa = "https://www.cpc.ncep.noaa.gov/data/indices/wksst9120.for"
respuesta_noaa = requests.get(url_noaa)

lineas = respuesta_noaa.text.splitlines()
datos_mar = []

# Expresión regular para capturar: Fecha (DDMMMYYYY) + TSM Niño 1+2 + Anomalía Niño 1+2
# Resuelve el problema de columnas pegadas cuando la anomalía lleva signo negativo
patron = re.compile(r"^\s*(\d{2}[A-Z]{3}\d{4})\s+(\d+\.\d+)([+-]?\s*\d+\.\d+)")

for linea in lineas[4:]: # Saltamos las 4 líneas de encabezado del archivo Fortran
    match = patron.match(linea)
    if match:
        fecha_str = match.group(1)
        tsm_nino12 = float(match.group(2))
        anom_nino12 = float(match.group(3).replace(" ", ""))
        datos_mar.append([fecha_str, tsm_nino12, anom_nino12])

## Uso de pandas para la creación del Dataset

In [3]:
# 2. ESTRUCTURACIÓN Y CONVERSIÓN A SEMANA EPIDEMIOLÓGICA (Domingo a Sábado - MINSA)
df_enfen = pd.DataFrame(datos_mar, columns=['Fecha_Str', 'TSM_Nino12', 'Anomalia_Nino12'])
df_enfen['Fecha'] = pd.to_datetime(df_enfen['Fecha_Str'], format='%d%b%Y')
domingo_epi = df_enfen['Fecha'] - pd.to_timedelta((df_enfen['Fecha'].dt.dayofweek + 1) % 7, unit='D')
miercoles_epi = domingo_epi + pd.Timedelta(days=3)
df_enfen['ano'] = miercoles_epi.dt.year.astype(int)
df_enfen['semana'] = ((miercoles_epi.dt.dayofyear - 1) // 7 + 1).astype(int)
display(df_enfen.head(10))  # Mostramos las primeras 10 filas para inspección

,Fecha_Str,TSM_Nino12,Anomalia_Nino12,Fecha,ano,semana
0,02SEP1981,20.6,-0.1,1981-09-02,1981,35
1,09SEP1981,20.1,-0.6,1981-09-09,1981,36
2,16SEP1981,19.7,-0.9,1981-09-16,1981,37
3,23SEP1981,20.3,-0.4,1981-09-23,1981,38
4,30SEP1981,20.0,-0.8,1981-09-30,1981,39
5,07OCT1981,20.1,-0.7,1981-10-07,1981,40
6,14OCT1981,20.0,-0.9,1981-10-14,1981,41
7,21OCT1981,20.4,-0.7,1981-10-21,1981,42
8,28OCT1981,20.7,-0.5,1981-10-28,1981,43
9,04NOV1981,20.8,-0.6,1981-11-04,1981,44


### Análisis Exploratorio Inicial

In [4]:
print(df_enfen.info())  # Información del DataFrame para verificar tipos de datos y memoria
print(df_enfen.describe())  # Estadísticas descriptivas para entender la distribución de los datos


<class 'pandas.DataFrame'>
RangeIndex: 2359 entries, 0 to 2358
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Fecha_Str        2359 non-null   str           
 1   TSM_Nino12       2359 non-null   float64       
 2   Anomalia_Nino12  2359 non-null   float64       
 3   Fecha            2359 non-null   datetime64[us]
 4   ano              2359 non-null   int32         
 5   semana           2359 non-null   int32         
dtypes: datetime64[us](1), float64(2), int32(2), str(1)
memory usage: 92.3 KB
None
        TSM_Nino12  Anomalia_Nino12                Fecha          ano  \
count  2359.000000      2359.000000                 2359  2359.000000   
mean     23.291479         0.036922  2004-04-07 00:00:00  2003.764731   
min      18.700000        -2.500000  1981-09-02 00:00:00  1981.000000   
25%      21.300000        -0.800000  1992-12-19 12:00:00  1992.000000   
50%      23.200000        -0.2000

Inicialmente se contempla un dataset crudo de 2359 registros en 4 columnas ya seleccionadas y procesadas en el código anterior que comprenden todo el periodo de 1981 al 2026, abarcando temperaturas de la costa de Piura entre 18.7 y 28.9 y señalando las Anomalías de temperatura generadas por el Fenómeno El Niño

## Procesamiento del dataset en crudo
El dataset original debe filtrarse para solo comprender el periodo del 2000 al 2025, en caso existan más de un registro hecho en una misma semana se usará el promediado, el mismo proceso se realiza para la temperatura superficial del mar.

In [5]:
# Filtramos el rango 2000-2025 y promediamos por si caen dos lecturas en una misma semana epidemiológica
df_enfen_semanal = df_enfen[(df_enfen['ano'] >= 2000) & (df_enfen['ano'] <= 2025)].groupby(['ano', 'semana']).agg({
    'TSM_Nino12': 'mean',
    'Anomalia_Nino12': 'mean'
}).reset_index().round(2)

print(f"Semanas oceánicas procesadas (Niño 1+2): {len(df_enfen_semanal)}")
print(df_enfen_semanal.tail())

Semanas oceánicas procesadas (Niño 1+2): 1357
       ano  semana  TSM_Nino12  Anomalia_Nino12
1352  2025      49        22.1             -0.3
1353  2025      50        22.3             -0.3
1354  2025      51        22.4             -0.5
1355  2025      52        22.9             -0.4
1356  2025      53        22.7             -0.9


In [6]:
df_enfen_semanal[df_enfen_semanal['Anomalia_Nino12'] > 0].head(10)  # Mostramos las primeras 10 semanas con anomalía positiva (Niño)

,ano,semana,TSM_Nino12,Anomalia_Nino12
7,2000,8,26.5,0.1
14,2000,15,26.0,0.2
15,2000,16,25.8,0.3
16,2000,17,25.4,0.2
62,2001,11,26.6,0.1
63,2001,12,27.5,1.1
64,2001,13,27.3,1.0
65,2001,14,27.1,1.0
66,2001,15,26.5,0.7
67,2001,16,26.1,0.6


## Exportación final del Dataset

In [7]:
# Guardar archivo CSV final en la carpeta 'Datasets' del proyecto
from pathlib import Path
ruta_datasets = next(
    (ruta / 'Datasets' for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / 'Datasets').is_dir()),
    None
)
if ruta_datasets is None:
    raise FileNotFoundError("No se encontró la carpeta 'Datasets' del proyecto.")

ruta_csv = ruta_datasets / 'temperatura_superficial_mar_piura_2000_2025_semanal.csv'
df_enfen_semanal.to_csv(ruta_csv, index=False)
print(f"Dataset guardado en: {ruta_csv}")

Dataset guardado en: c:\Users\fabri\OneDrive\Desktop\PROYECTO-IA\Datasets\temperatura_superficial_mar_piura_2000_2025_semanal.csv
